# 自然语言处理
- torch.utils.data 用于数据加载和预处理的核心模块。它提供了构建高效数据管道的工具，主要包括以下组件：
- torch.nn.Embedding 用于创建词嵌入

## torch.utils.data

| 类               | 用途                                            |
| --------------- | --------------------------------------------- |
| `Dataset`       | 抽象基类，自定义数据集需继承此类并实现 `__len__` 和 `__getitem__` |
| `TensorDataset` | 包装张量作为数据集                                     |
| `ConcatDataset` | 连接多个数据集                                       |
| `ChainDataset`  | 链式组合数据集（用于迭代器风格）                              |
| `Subset`        | 数据集的子集                                        |
| `random_split`  | 随机将数据集划分为非重叠子集                                |


### 关键特性
- 自动批处理：DataLoader 自动将样本组合成 batch
- 多进程加速：num_workers > 0 启用多进程数据加载
- 内存固定：pin_memory=True 加速 GPU 数据传输
- 自定义采样：通过 sampler 参数实现复杂采样策略
- 数据增强：通常在 Dataset.__getitem__ 中实现

In [ ]:
from  torch.utils.data import DataLoader,Dataset

# 基本用法
loader = DataLoader(
    dataset, # Dataset实例
    batch_size=32, # 每批样本数
    shuffle=True,
    num_workers = 4,
    pin_memory=True, # 是否将数据固定在CUDA内存中
    drop_last = False, # 是否丢弃最后不完整的batch
    collate_fn = None, # 自定义batch组装函数
)

| Sampler                 | 说明                   |
| ----------------------- | -------------------- |
| `SequentialSampler`     | 顺序采样（默认）             |
| `RandomSampler`         | 随机采样                 |
| `SubsetRandomSampler`   | 子集随机采样               |
| `WeightedRandomSampler` | 加权随机采样（处理类别不平衡）      |
| `BatchSampler`          | 包装其他Sampler返回batch索引 |
| `DistributedSampler`    | 分布式训练采样              |


In [8]:
import torch
from  torch.utils.data import Dataset,DataLoader,random_split

class MyDataset(Dataset):
    def __init__(self,data,labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self,idx):
        return self.data[idx],self.labels[idx]
# 创建数据
data = torch.randn(1000,10)
print(data[0])
labels = torch.randint(0,5,(1000,))
print(labels)
dataset = MyDataset(data,labels)

# 划分训练、验证集
train_set,val_set = random_split(dataset,[800,200])

# 创建DataLoader
train_loader = DataLoader(train_set,batch_size=32,shuffle=True,num_workers=2)
val_loader = DataLoader(train_set,batch_size=32,shuffle=False)

# 训练循环
for batch_data,batch_labels in train_loader:
    pass


tensor([-0.2046,  0.7062,  1.9111,  0.0048,  0.9425, -0.6899, -0.2719, -0.7575,
         0.0086,  0.8423])
tensor([1, 0, 2, 4, 2, 0, 2, 2, 0, 2, 3, 1, 0, 3, 4, 4, 2, 0, 1, 3, 0, 0, 3, 2,
        3, 4, 0, 3, 2, 2, 3, 0, 2, 4, 1, 3, 1, 4, 3, 3, 3, 0, 0, 1, 4, 3, 3, 0,
        1, 1, 3, 4, 0, 4, 0, 4, 2, 0, 0, 2, 3, 2, 3, 0, 0, 1, 3, 1, 4, 0, 1, 0,
        4, 1, 1, 1, 0, 1, 1, 0, 2, 2, 3, 0, 0, 2, 4, 4, 3, 4, 3, 0, 0, 4, 1, 2,
        2, 3, 1, 2, 3, 3, 3, 4, 2, 0, 2, 0, 3, 4, 3, 4, 3, 4, 2, 0, 2, 0, 3, 4,
        4, 1, 0, 1, 4, 1, 2, 3, 0, 4, 3, 3, 0, 4, 2, 2, 0, 3, 4, 4, 1, 0, 3, 3,
        2, 0, 0, 0, 1, 4, 4, 4, 0, 3, 2, 4, 2, 2, 2, 2, 2, 3, 3, 1, 2, 2, 1, 0,
        3, 1, 1, 2, 0, 0, 4, 3, 3, 4, 0, 2, 3, 1, 2, 1, 1, 2, 2, 1, 3, 2, 1, 4,
        2, 1, 0, 2, 3, 4, 4, 1, 0, 0, 4, 3, 3, 2, 3, 3, 1, 0, 3, 2, 2, 3, 2, 2,
        1, 1, 3, 0, 1, 0, 1, 3, 0, 0, 3, 2, 2, 3, 4, 3, 0, 0, 1, 2, 1, 2, 0, 3,
        4, 1, 1, 4, 3, 4, 1, 2, 0, 1, 4, 1, 3, 1, 1, 2, 4, 3, 4, 2, 3, 3, 1, 0,
        3, 4,

| 函数                   | 说明                   | 示例                             |
| -------------------- | -------------------- | ------------------------------ |
| `torch.randint`      | 均匀分布整数 `[low, high)` | `torch.randint(0, 10, (3,))`   |
| `torch.rand`         | 均匀分布浮点数 `[0, 1)`     | `torch.rand(3, 3)`             |
| `torch.randn`        | 标准正态分布浮点数            | `torch.randn(3, 3)`            |
| `torch.randperm`     | `0` 到 `n-1` 的随机排列    | `torch.randperm(10)`           |
| `torch.randint_like` | 同形状随机整数张量            | `torch.randint_like(x, 0, 10)` |


## torch.nn.Embedding() 用于创建词嵌入

- nn.Embedding 是一个查找表（Lookup Table），将离散的整数索引映射为连续的稠密向量。这是深度学习处理离散数据（如词语、类别）的基础组件。

In [ ]:
torch.nn.Embedding(
    num_embeddings,    # 词汇表大小（最大索引 + 1）
    embedding_dim,     # 每个词向量的维度
    padding_idx=None,  # 指定填充符索引，该位置向量恒为0且不参与梯度更新
    max_norm=None,     # 若设置，向量会被归一化到该范数
    norm_type=2.0,     # 使用的范数类型（默认L2）
    scale_grad_by_freq=False,  # 是否按词频缩放梯度
    sparse=False,      # 是否使用稀疏梯度（仅对特定优化器有效）
    _weight=None       # 预初始化权重
)

### 工作原理

In [ ]:
输入: 整数索引张量  →  查找表  →  输出: 稠密向量张量

例如: [[1, 5, 3]]   →  Embedding  →  [[[0.2, 0.5], [0.1, 0.9], [0.4, 0.3]]]
       (batch, seq)                (batch, seq, embedding_dim)

In [10]:
import torch
import torch.nn as nn

# 基础用法
embedding = nn.Embedding(num_embeddings=10,embedding_dim=3) # 10个词，每个词3个维度

# 输入：batch=2，seq_len=4 的句子（每个数字代表词的ID）
input_ids = torch.LongTensor([[1, 2, 4, 5], [4, 3, 2, 9]])
output = embedding(input_ids) # shape:(2,4,3) 

print(f"输入形状：{input_ids.shape}")
print(f"输出形状：{output.shape}")
print(f"输出示例:{output[0]}") # 第一行句子的4个词的词向量，每个词的词向量为3个浮点数

输入形状：torch.Size([2, 4])
输出形状：torch.Size([2, 4, 3])
输出示例:tensor([[ 1.6129, -0.7388, -0.4995],
        [ 0.7525,  0.9083,  0.5595],
        [ 0.6446,  0.2752, -0.1232],
        [ 1.0815, -0.1500, -0.3363]], grad_fn=<SelectBackward0>)


In [16]:
# 2 带padding_idx 推荐用于变长序列
embedding_pad = nn.Embedding(10,3,padding_idx=0)
# 索引0对应的向量始终为0，且不会更新
padded_input = torch.LongTensor([[1, 2, 0, 0], [3, 0, 0, 0]]) # 0是填充符
padded_output = embedding_pad(padded_input)

print(f"padding位置是否为0：{torch.all(padded_output[1,1:] == 0)}") # 第一行的索引1之后的都为0

padding位置是否为0：True


In [18]:
# 查看并修改权重
print(f"权重矩阵形状:{embedding.weight.shape}")
# 可用预训练词向量初始化
pretrained = torch.randn(10, 3)
embedding.weight.data.copy_(pretrained)

权重矩阵形状:torch.Size([10, 3])


tensor([[ 0.6436,  2.7299,  2.5689],
        [ 1.6423, -1.7324,  0.0437],
        [-2.6349,  1.1544, -0.2907],
        [ 0.7892, -0.5135, -0.2460],
        [-1.3753,  0.7271, -0.5020],
        [-0.8859, -0.4445, -2.2076],
        [ 1.1060, -0.3823, -2.1067],
        [-1.4264,  0.4496,  0.6718],
        [ 0.4974,  0.2917,  0.6629],
        [ 0.2629,  1.4408, -2.9132]])

In [20]:
# 4. 获取特定词的向量（类似字典查询）
word_vec = embedding(torch.LongTensor([5]))  # 获取索引5的词向量
print(word_vec)

tensor([[-0.8859, -0.4445, -2.2076]], grad_fn=<EmbeddingBackward0>)


In [ ]:
# 方式1: nn.Module（有参数，可学习）
embedding_layer = nn.Embedding(1000, 128)
output = embedding_layer(input_ids)
print(output)

# 方式2: F.embedding（无参数，需手动提供权重）
import torch.nn.functional as F
weight = torch.randn(1000, 128)  # 预训练或自定义
output = F.embedding(input_ids, weight)
print(output)

## nn.Embedding在Transformer中的使用

在 Transformer 中承担两个核心角色：
- Token Embedding 
- Position Embedding。

### Token Embedding(词嵌入)
- 将输入序列的离散token ID映射为稠密向量


In [31]:
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self,vocab_size,d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.d_model = d_model
    def forward(self,x):
        # 输入 x:[batch_size,seq_len]
        # 乘以 sqrt(d_model) 是attention is all you need论文中的缩放技巧
        return self.embedding(x) * (self.d_model ** 0.5)

    # 示例 
vocab_size,d_model = 30000,512
token_emb = TokenEmbedding(vocab_size,d_model)
input_ids = torch.randint(0,vocab_size,(2,20)) # batch=2,seq=20
token_embeddings = token_emb(input_ids)
print(len(token_embeddings[1]))

20


### position embedding（位置编码)


In [ ]:
# 可学习的 Position Embedding（BERT 风格）
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=512):
        super().__init__()
        # 为每个位置学习一个向量
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        
    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        batch_size, seq_len, _ = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        # 广播到 batch_size: [1, seq_len] → [batch_size, seq_len]
        pos_emb = self.pos_embedding(positions)  # [1, seq_len, d_model]
        return x + pos_emb  # 广播相加

In [ ]:
# 固定的 Sinusoidal 编码（原版 Transformer）
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        
        # 分母项: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维
        
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_seq_len, d_model]
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
# 完整的Transformer Embedding 层
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len=512, dropout=0.1, 
                 pos_encoding='learnable'):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.scale = d_model ** 0.5
        
        if pos_encoding == 'learnable':
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.use_sinusoidal = False
        else:
            self.pos_emb = SinusoidalPositionalEncoding(d_model, max_seq_len)
            self.use_sinusoidal = True
            
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Token Embedding + 缩放
        tok_emb = self.token_emb(x) * self.scale  # [batch, seq, d_model]
        
        # Position Embedding
        if self.use_sinusoidal:
            emb = self.pos_emb(tok_emb)
        else:
            positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
            pos_emb = self.pos_emb(positions)  # [1, seq, d_model]
            emb = tok_emb + pos_emb
            
        return self.dropout(emb)

# 使用
embedding_layer = TransformerEmbedding(
    vocab_size=30000, 
    d_model=512,
    pos_encoding='learnable'  # 或 'sinusoidal'
)

### 关键对比

| 特性       | 可学习 Position Embedding    | Sinusoidal Encoding |
| -------- | ------------------------- | ------------------- |
| **来源**   | BERT、GPT、ViT              | 原版 Transformer      |
| **参数**   | 需要学习（max\_len × d\_model） | 无参数，预计算             |
| **外推性**  | 差（超过 max\_len 需插值）        | 好（可处理任意长度）          |
| **实现**   | `nn.Embedding`            | `register_buffer`   |
| **相对位置** | 隐式学习                      | 显式编码，便于捕捉相对位置       |


In [ ]:
Input IDs [batch, seq] 
    ↓
Token Embedding (nn.Embedding) → [batch, seq, d_model] × √d_model
    ↓
Position Embedding (nn.Embedding 或 Sinusoidal) → 相加
    ↓
Dropout
    ↓
输入到 Multi-Head Attention

关键注意事项
- 输入类型：必须是 LongTensor 或 Long，整数索引
- 索引范围：0 ≤ index < num_embeddings，越界会报错
- 梯度更新：padding_idx 指定的索引不会参与梯度计算
- 初始化：默认使用 N(0,1)  初始化，实际中常用 Xavier 或预训练向量

# 高级模型架构
- torch.nn.Transformer: Transformer模型的实现
- torch.nn.MultiheadAttention:多头注意力机制的实现
- torch.utils.cpp_extension:允许使用C++或CUDA扩展pytorch

## torch.nn.Transformer 
- 是 PyTorch 实现的完整 Transformer 模型，基于论文 "Attention Is All You Need" (Vaswani et al., 2017)

In [ ]:
import torch
import torch.nn as nn

# 创建 Transformer 模型
transformer_model = nn.Transformer(
    d_model=512,           # 模型维度
    nhead=8,               # 注意力头数
    num_encoder_layers=6,  # 编码器层数
    num_decoder_layers=6,  # 解码器层数
    dim_feedforward=2048,  # 前馈网络维度
    dropout=0.1,           # Dropout
    batch_first=False      # 序列长度在第一位
)

# 输入数据 (seq_len, batch_size, d_model)
src = torch.rand(10, 32, 512)  # 源序列: 10个token, batch=32
tgt = torch.rand(20, 32, 512)  # 目标序列: 20个token, batch=32

# 前向传播
out = transformer_model(src, tgt)
# out.shape: (20, 32, 512) - 目标序列长度, batch, 特征维度

关键组件
1. 编码器 (Encoder)
- 由 num_encoder_layers 个 TransformerEncoderLayer 堆叠
- 每个层包含：多头自注意力 + 前馈网络 + LayerNorm + 残差连接
2. 解码器 (Decoder)
- 由 num_decoder_layers 个 TransformerDecoderLayer 堆叠
- 每个层包含：
    - Masked 多头自注意力（防止看到未来token）
    - 多头交叉注意力（关注编码器输出）
    - 前馈网络
3. 注意力掩码 (Masks)

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """位置编码"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0).transpose(0, 1))
    
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, 
                 num_layers=6, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=dropout
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None,
                src_padding_mask=None, tgt_padding_mask=None):
        # 嵌入 + 位置编码
        src = self.embedding(src) * math.sqrt(self.d_model)
        tgt = self.embedding(tgt) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)
        
        # Transformer
        output = self.transformer(
            src, tgt,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        
        return self.fc_out(output)

# 使用示例
vocab_size = 10000
model = TransformerModel(vocab_size)

# 模拟数据
src = torch.randint(0, vocab_size, (10, 32))  # (seq_len, batch)
tgt = torch.randint(0, vocab_size, (20, 32))

# 生成因果掩码
tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(0))

output = model(src, tgt, tgt_mask=tgt_mask)
print(output.shape)  # (20, 32, vocab_size)

### 单独使用 Encoder 或 Decoder

In [ ]:
# 仅使用编码器（如 BERT 风格）
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)

# 仅使用解码器（如 GPT 风格）
decoder_layer = nn.TransformerDecoderLayer(d_model=512, nhead=8)
transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=6)

| 特性        | `torch.nn.Transformer` | Hugging Face |
| --------- | ---------------------- | ------------ |
| 预训练权重     | 无                      | 提供多种预训练模型    |
| Tokenizer | 需自行实现                  | 内置完善         |
| 易用性       | 需要更多手动配置               | 高级API，开箱即用   |
| 灵活性       | 完全可控                   | 封装较高         |
| 适用场景      | 研究/自定义架构               | 快速部署/生产环境    |


## torch.utils.cpp_extension
- 这是 PyTorch 中用于编译和加载 C++/CUDA 扩展的重要工具。

torch.utils.cpp_extension 是 PyTorch 提供的实用工具模块，主要用于：
- 即时编译 (JIT) C++ 和 CUDA 代码
- 简化扩展构建 流程
- 加载动态链接库 作为 Python 模块

# 优化和调试
- torch.cuda.amp: 提供了自动混合精度训练的功能，以提高型号和效率
- torchsummary：提供模型架构和参数的详细总结（非官方工具）

## torch.cuda.amp 
- 是 PyTorch 中用于 自动混合精度（Automatic Mixed Precision, AMP） 训练的模块。
- 它可以在支持的 GPU 上自动选择使用 FP16（半精度浮点数）或 FP32（单精度浮点数）进行计算，从而显著减少显存占用并加速训练，同时保持模型精度。

| 组件                          | 作用                  |
| --------------------------- | ------------------- |
| `torch.cuda.amp.autocast`   | 上下文管理器，自动选择精度进行前向传播 |
| `torch.cuda.amp.GradScaler` | 梯度缩放器，防止 FP16 梯度下溢  |
